# Role-boundary non-forgeability

Demonstrates how a ChatML-style template lets user-controlled content forge an assistant control marker, then validates the witness certificate.


The code cell below builds a minimal in-memory artifact, runs the real PromptABI checker, and asserts the proof obligation.


In [ ]:
from promptabi import ChatTemplateSymbolicBounds, parse_hf_chat_template_config
from promptabi import analyze_role_boundary_nonforgeability, prove_role_boundary_nonforgeability
from promptabi.proof_sketches import ProofOutcome

parsed = parse_hf_chat_template_config({
    "chat_template": (
        "{% for message in messages %}"
        "<|im_start|>{{ message['role'] }}\n"
        "{{ message['content'] }}<|im_end|>\n"
        "{% endfor %}"
    ),
    "additional_special_tokens": ["<|im_start|>", "<|im_end|>"],
})
report = analyze_role_boundary_nonforgeability(
    parsed,
    bounds=ChatTemplateSymbolicBounds(max_messages=1, max_tools=0, max_loop_iterations=1, max_paths=8),
)
sketch = prove_role_boundary_nonforgeability(report)

assert report.findings
assert sketch.outcome is ProofOutcome.COUNTEREXAMPLE
assert sketch.passed
sketch.to_dict()
